William Diallo Technical Assessment

In [0]:
import random, json, datetime as dt

catalog, schema, volume = "tech_assessment", "bean_counter", "landing"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
base = f"/Volumes/{catalog}/{schema}/{volume}"

random.seed(42)
stores = ["LDN01", "MAN01", "LDS01", "BRS01", "WEB01"]
products = [(f"P{i:03d}", n, c, p) for i, (n, c, p) in enumerate([
    ("Flat White", "Hot Drinks", 3.20), ("Latte", "Hot Drinks", 3.40),
    ("Cappuccino", "Hot Drinks", 3.30), ("Iced Mocha", "Cold Drinks", 4.10),
    ("Croissant", "Food", 2.50), ("Banana Bread", "Food", 2.80),
    ("Beans 1kg", "Retail", 14.00), ("Reusable Cup", "Retail", 9.50)], start=1)]

# Products (clean reference data)
with open(f"{base}/products.csv", "w") as f:
    f.write("product_id,product_name,category,unit_price\n")
    for p in products:
        f.write(",".join(map(str, p)) + "\n")

# Customers (JSON lines, batch 1 and a later batch with a new column)
def customers(start, n, extra=False):
    rows = []
    for i in range(start, start + n):
        r = {"customer_id": f"C{i:05d}",
             "email": f"user{i}@example.com",
             "city": random.choice(["London", "Manchester", "Leeds", "Bristol", None]),
             "signup_date": str(dt.date(2024, 1, 1) + dt.timedelta(days=random.randint(0, 600))),
             "updated_at": str(dt.datetime(2025, 1, 1) + dt.timedelta(hours=random.randint(0, 2000)))}
        if extra:
            r["marketing_opt_in"] = random.choice([True, False])
        rows.append(r)
    return rows

def write_jsonl(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

In [0]:
import os
for d in ["orders", "customers"]:
    os.makedirs(f"{base}/{d}", exist_ok=True)

write_jsonl(f"{base}/customers/customers_batch1.json", customers(1, 500))
# batch 2: 100 updated customers (changes) + 100 new, plus a NEW column
write_jsonl(f"{base}/customers/customers_batch2.json", customers(401, 200, extra=True))

# Orders: 3 daily files, with duplicates, nulls, bad values, mixed date formats
def orders(day, n, start_id):
    rows = []
    for i in range(n):
        oid = start_id + i
        p = random.choice(products)
        qty = random.choice([1, 1, 1, 2, 3, -1, 0]) # bad quantities
        cust = random.choice([f"C{random.randint(1, 550):05d}", "", None])  # missing customers
        ts = dt.datetime.combine(day, dt.time(random.randint(7, 20), random.randint(0, 59)))
        fmt = random.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"]) # mixed formats
        rows.append([f"O{oid:06d}", cust, random.choice(stores), p[0], qty,
                     p[3] if random.random() > 0.03 else "", ts.strftime(fmt)])
        if random.random() < 0.05: # duplicates
            rows.append(rows[-1])
    return rows

for n, day in enumerate([dt.date(2025, 3, 1), dt.date(2025, 3, 2), dt.date(2025, 3, 3)]):
    with open(f"{base}/orders/orders_{day}.csv", "w") as f:
        f.write("order_id,customer_id,store_id,product_id,quantity,unit_price,order_ts\n")
        for r in orders(day, 2000, n * 10000):
            f.write(",".join(map(str, r)) + "\n")

print("Landing files:", os.listdir(f"{base}/orders"), os.listdir(f"{base}/customers"))

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp
    )
from pyspark.sql import SparkSession
checkpoints_base= f"{BASE}/_checkpoints"
schemas_base = f"{BASE}/_schemas"
orders_base = f"{BASE}/orders"
customers_base = f"{BASE}/customers"
orders_delta_table = f"{catalog}.{schema}.bronze_orders"
customers_delta_table = f"{catalog}.{schema}.bronze_customers"

# 1.1 - Ingests orders via autoloader
orders_read = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{schemas_base}/orders")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes","true")
    .load(orders_base)
    )

# 1.2 - provides the ingestion metadata
orders_bronze = (
    orders_read
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

orders_write = (
    orders_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoints_base}/orders")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(orders_delta_table)
    )
orders_write.awaitTermination()

display(spark.read.table(orders_delta_table))



In [0]:
#dbutils.fs.rm(f"{BASE}", recurse=True)
#orders_read.printSchema()
#display(spark.read.text(f"{schemas_base}/orders"))